# 2층 신경망 순전파 및 역전파 연쇄 법칙(Chain Rule) 유도

## 1. 네트워크 구조 및 표기법
- **구조**: 입력층(2) → 은닉층(2, Sigmoid) → 출력층(1, Sigmoid)
- **손실 함수**: Binary Cross-Entropy (BCE)
$$L(y, \hat{y}) = -\left(y \log(\hat{y}) + (1 - y)\log(1 - \hat{y})\right)$$

### 텐서 Shape 규격
| 변수 | 의미 | Shape |
| :--- | :--- | :--- |
| $x$ | 입력 벡터 | `(2,)` |
| $W_1$ | 입력→은닉 가중치 행렬 | `(2, 2)` |
| $b_1$ | 은닉층 편향 벡터 | `(2,)` |
| $z_1$ | 은닉층 선형 결합 ($x W_1 + b_1$) | `(2,)` |
| $a_1$ | 은닉층 활성화 출력 ($\sigma(z_1)$) | `(2,)` |
| $W_2$ | 은닉→출력 가중치 벡터 | `(2,)` |
| $b_2$ | 출력층 편향 스칼라 | `()` |
| $z_2$ | 출력층 선형 결합 ($a_1 \cdot W_2 + b_2$) | `()` |
| $\hat{y}$ (`y_pred`) | 예측 확률 ($\sigma(z_2)$) | `()` |
| $y$ (`y_true`) | 실제 정답 레이블 ($y = 1$) | `()` |


## 2. 일반 수식 유도 (순전파 & 역전파)

### 2.1 순전파 (Forward Pass)
1. $z_1 = x W_1 + b_1 \quad \text{shape: } (2,)$
2. $a_1 = \sigma(z_1) = \frac{1}{1 + e^{-z_1}} \quad \text{shape: } (2,)$
3. $z_2 = a_1 \cdot W_2 + b_2 \quad \text{shape: } ()$
4. $\hat{y} = \sigma(z_2) = \frac{1}{1 + e^{-z_2}} \quad \text{shape: } ()$
5. $L = -\left(y\log(\hat{y}) + (1-y)\log(1-\hat{y})\right) \quad \text{shape: } ()$

### 2.2 역전파 (Backward Pass — 연쇄 법칙)
출력층에서 입력층 방향으로 미분을 전파합니다.

1. **손실에 대한 예측값 기울기**:
$$\frac{\partial L}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1 - y}{1 - \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1 - \hat{y})} \quad \text{shape: } ()$$

2. **출력층 선형 결합 $z_2$에 대한 기울기** (Sigmoid 도함수 $\sigma'(z_2) = \hat{y}(1-\hat{y})$ 결합):
$$\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2} = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y}) = \hat{y} - y \quad \text{shape: } ()$$

3. **출력층 가중치 $W_2$ 및 편향 $b_2$에 대한 기울기**:
$$\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial z_2} \cdot a_1 \quad \text{shape: } (2,), \qquad \frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2} \quad \text{shape: } ()$$

4. **은닉층 활성화 출력 $a_1$에 대한 기울기**:
$$\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} \cdot W_2 \quad \text{shape: } (2,)$$

5. **은닉층 선형 결합 $z_1$에 대한 기울기**:
$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot a_1 \odot (1 - a_1) \quad \text{shape: } (2,)$$

6. **입력층 가중치 $W_1$ 및 편향 $b_1$에 대한 기울기**:
$$\frac{\partial L}{\partial W_1} = x^T \otimes \frac{\partial L}{\partial z_1} = \text{outer}\left(x, \frac{\partial L}{\partial z_1}\right) \quad \text{shape: } (2, 2), \qquad \frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1} \quad \text{shape: } (2,)$$


## 3. 필수 고정 예제 손계산 과정
*(참고: 과제 명세의 `y_true =` 항목은 이진 분류 표준 정답인 `y_true = 1`로 설정하여 계산합니다.)*

- **입력 및 파라미터**:
  - $x = [1, 0]$ `shape: (2,)`
  - $W_1 = \begin{bmatrix} 0.1 & 0.2 \\ 0.3 & 0.4 \end{bmatrix}$ `shape: (2, 2)`, $b_1 = [0, 0]$ `shape: (2,)`
  - $W_2 = [0.5, 0.6]$ `shape: (2,)`, $b_2 = 0$ `shape: ()`
  - $y_{\text{true}} = 1$ `shape: ()`

### 3.1 순전파 손계산
1. $z_1 = [1, 0] \begin{bmatrix} 0.1 & 0.2 \\ 0.3 & 0.4 \end{bmatrix} + [0, 0] = [0.1, 0.2]$ `shape: (2,)`
2. $a_1 = [\sigma(0.1), \sigma(0.2)] = \left[\frac{1}{1+e^{-0.1}}, \frac{1}{1+e^{-0.2}}\right] \approx [0.524979, 0.549834]$ `shape: (2,)`
3. $z_2 = 0.524979 \times 0.5 + 0.549834 \times 0.6 + 0 = 0.262490 + 0.329900 = 0.592390$ `shape: ()`
4. $\hat{y} = \sigma(0.592390) = \frac{1}{1 + e^{-0.592390}} \approx 0.643913$ `shape: ()`
5. $L = -\left(1 \cdot \ln(0.643913) + 0\right) \approx 0.440191$ `shape: ()`

### 3.2 역전파 손계산
1. $\frac{\partial L}{\partial \hat{y}} = -\frac{1}{0.643913} \approx -1.553004$ `shape: ()`
2. $\frac{\partial L}{\partial z_2} = \hat{y} - y = 0.643913 - 1 = -0.356087$ `shape: ()`
3. $\frac{\partial L}{\partial W_2} = (-0.356087) \cdot [0.524979, 0.549834] \approx [-0.186938, -0.195789]$ `shape: (2,)`
4. $\frac{\partial L}{\partial a_1} = (-0.356087) \cdot [0.5, 0.6] \approx [-0.178043, -0.213652]$ `shape: (2,)`
5. $\frac{\partial L}{\partial z_1} = [-0.178043, -0.213652] \odot [0.524979(1-0.524979), 0.549834(1-0.549834)]$
   $= [-0.178043 \times 0.249376, -0.213652 \times 0.247517] \approx [-0.044400, -0.052882]$ `shape: (2,)`
6. $\frac{\partial L}{\partial W_1} = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \begin{bmatrix} -0.044400 & -0.052882 \end{bmatrix} = \begin{bmatrix} -0.044400 & -0.052882 \\ 0.000000 & 0.000000 \end{bmatrix}$ `shape: (2, 2)`


In [1]:
import numpy as np

np.random.seed(42)

def sigmoid(z: np.ndarray) -> np.ndarray:
    """시그모이드 활성화 함수: 1 / (1 + exp(-z))."""
    return 1.0 / (1.0 + np.exp(-z))

# 1. 고정 예제 입력 및 가중치 정의
x = np.array([1.0, 0.0], dtype=np.float64)                  # shape: (2,)
W1 = np.array([[0.1, 0.2], [0.3, 0.4]], dtype=np.float64)   # shape: (2, 2)
b1 = np.array([0.0, 0.0], dtype=np.float64)                 # shape: (2,)
W2 = np.array([0.5, 0.6], dtype=np.float64)                 # shape: (2,)
b2 = 0.0                                                    # shape: ()
y_true = 1.0                                                # shape: ()

# 2. 순전파 (Forward Pass)
z1 = x @ W1 + b1                                            # shape: (2,)
a1 = sigmoid(z1)                                            # shape: (2,)
z2 = float(a1 @ W2 + b2)                                    # shape: ()
y_pred = float(sigmoid(z2))                                 # shape: ()
loss = -(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))

print("=== 순전파 중간 체크포인트 ===")
print(f"z1     {z1.shape}: {np.round(z1, 6)}")
print(f"a1     {a1.shape}: {np.round(a1, 6)}")
print(f"z2     (): {z2:.6f}")
print(f"y_pred (): {y_pred:.6f}")
print(f"Loss   (): {loss:.6f}")

# 3. 역전파 (Backward Pass)
dL_dy_pred = -y_true / y_pred + (1.0 - y_true) / (1.0 - y_pred)
dL_dz2 = y_pred - y_true
dL_dW2 = dL_dz2 * a1
dL_db2 = dL_dz2
dL_da1 = dL_dz2 * W2
dL_dz1 = dL_da1 * a1 * (1.0 - a1)
dL_dW1 = np.outer(x, dL_dz1)
dL_db1 = dL_dz1

print("\n=== 역전파 기울기 결과 ===")
print(f"dL/dy_pred (): {dL_dy_pred:.6f}")
print(f"dL/dz2     (): {dL_dz2:.6f}")
print(f"dL/dW2     {dL_dW2.shape}: {np.round(dL_dW2, 6)}")
print(f"dL/da1     {dL_da1.shape}: {np.round(dL_da1, 6)}")
print(f"dL/dz1     {dL_dz1.shape}: {np.round(dL_dz1, 6)}")
print(f"dL/dW1     {dL_dW1.shape}:\n{np.round(dL_dW1, 6)}")

# 4. 손계산 결과와 소수점 4자리(1e-4) 일치 검증
hand_z1 = np.array([0.1, 0.2])
hand_a1 = np.array([0.524979, 0.549834])
hand_y_pred = 0.643913
hand_dL_dW2 = np.array([-0.186938, -0.195789])
hand_dL_dW1 = np.array([[-0.044400, -0.052882], [0.0, 0.0]])

assert np.allclose(z1, hand_z1, atol=1e-4)
assert np.allclose(a1, hand_a1, atol=1e-4)
assert np.isclose(y_pred, hand_y_pred, atol=1e-4)
assert np.allclose(dL_dW2, hand_dL_dW2, atol=1e-4)
assert np.allclose(dL_dW1, hand_dL_dW1, atol=1e-4)
print("\n[검증 성공] 손계산 결과와 NumPy 계산이 소수점 4자리 이상 일치합니다.")


=== 순전파 중간 체크포인트 ===
z1     (2,): [0.1 0.2]
a1     (2,): [0.524979 0.549834]
z2     (): 0.592390
y_pred (): 0.643913
Loss   (): 0.440191

=== 역전파 기울기 결과 ===
dL/dy_pred (): -1.553004
dL/dz2     (): -0.356087
dL/dW2     (2,): [-0.186938 -0.195789]
dL/da1     (2,): [-0.178043 -0.213652]
dL/dz1     (2,): [-0.0444   -0.052882]
dL/dW1     (2, 2):
[[-0.0444   -0.052882]
 [-0.       -0.      ]]

[검증 성공] 손계산 결과와 NumPy 계산이 소수점 4자리 이상 일치합니다.
